In [1]:
# Fix Windows symlink issue
import os
os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
print("âœ… Disabled HuggingFace symlinks for Windows compatibility")

âœ… Disabled HuggingFace symlinks for Windows compatibility


# RAG on ASQA Long-Form QA Dataset

This notebook implements and compares two RAG baselines on the ASQA dataset:
1. **Normal RAG**: Standard retrieve-and-generate pipeline
2. **RAG + LLM Filter**: Two-stage filtering (context + answer)

ASQA focuses on ambiguous questions requiring long-form answers (50-200 words).

## 1. Setup and Imports

In [2]:
import sys
sys.path.append('..')

import os
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
import dotenv

# Load environment variables
dotenv.load_dotenv()

# Import RAG system components
from src.config import RAGConfig
from src.rag_system import RAGSystem
from src.data.loader import ASQALoader
from src.filtering.llm_filter import LLMFilterPipeline

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\src\filtering\llm_filter.py:39: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 2. Load ASQA Dataset

In [3]:
# Initialize ASQA loader
loader = ASQALoader()

# Load data
train_path = "../data/asqa/train.csv"
dev_path = "../data/asqa/dev.csv"

df_train, df_dev = loader.load_data(train_path, dev_path)

# Display statistics
stats = loader.get_statistics()
print("\nDataset Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Loading ASQA data...
Loaded 4353 training samples
Loaded 948 dev samples

Answer length statistics (words):
  Mean: 73.3
  Median: 68.0
  Min: 10
  Max: 491

Dataset Statistics:
  train_samples: 4353
  train_avg_docs_per_question: 4.160578911095796
  train_avg_answer_length: 73.33195497358143
  dev_samples: 948
  dev_avg_docs_per_question: 4.836497890295359
  dev_avg_answer_length: 57.857594936708864
  valid_samples: 948
  valid_avg_docs_per_question: 4.836497890295359


In [4]:
# Examine sample
sample = df_train.iloc[0]
print("Sample Question:")
print(sample['question'])
print("\nSample Answer (long-form):")
print(sample['answer'])
print(f"\nAnswer length: {len(sample['answer'].split())} words")
print(f"Number of context documents: {len(sample['docs'])}")

Sample Question:
When does the new bunk'd come out?

Sample Answer (long-form):
The new bunk'd episode 41 comes out on April 21, 2017, episode 42 comes out on April 28, 2017 and episode 42 is due to come out on May 24, 2017. 

Answer length: 31 words
Number of context documents: 4


## 3. Configure RAG System for Long-Form Generation

In [5]:
# Configure for ASQA (long-form answers)
config = RAGConfig(
    # Models
    encoder_model="sentence-transformers/all-MiniLM-L6-v2",
    generator_model="google/flan-t5-base",   # flan-t5-base: 3x smaller than large, fits in RAM

    # Paths
    models_dir="../models",
    output_dir="./rag_output_asqa",
    train_data_path=train_path,
    valid_data_path=dev_path,

    # Retriever training
    retriever_epochs=5,
    retriever_batch_size=16,
    retriever_lr=2e-5,
    retriever_patience=3,

    # Generator training
    generator_epochs=3,
    generator_batch_size=4,               # Increased from 2 (smaller model allows larger batch)
    generator_lr=5e-5,
    generator_max_input_tokens=512,
    generator_max_target_tokens=128,      # ASQA avg answer = 73 words; 128 tokens is sufficient

    # Checkpoint settings — sub-epoch saving so a crash loses at most N steps
    retriever_checkpoint_steps=100,       # ~90 sec between retriever checkpoints
    generator_checkpoint_steps=200,       # ~45 min between generator checkpoints
    checkpoint_dir=Path("./rag_output_asqa/checkpoints"),

    # Retrieval
    top_k=5,

    # Generation (inference only)
    max_new_tokens=256,

    # Device
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Configuration:")
print(f"  Encoder: {config.encoder_model}")
print(f"  Generator: {config.generator_model}")
print(f"  Max target tokens: {config.generator_max_target_tokens}")
print(f"  Generator batch size: {config.generator_batch_size}")
print(f"  Checkpoint dir: {config.checkpoint_dir}")
print(f"  Device: {config.device}")

Configuration:
  Encoder: sentence-transformers/all-MiniLM-L6-v2
  Generator: google/flan-t5-base
  Max target tokens: 128
  Generator batch size: 4
  Checkpoint dir: rag_output_asqa\checkpoints
  Device: cpu


## 4. Initialize RAG System

In [6]:
# Initialize RAG system
rag = RAGSystem(config=config)

# Use ASQA loader instead of default HotpotQA loader
rag.data_loader = loader


Initializing RAG System
Device: cpu
Encoder: sentence-transformers/all-MiniLM-L6-v2
Generator: google/flan-t5-base
Models directory: ..\models
Output directory: rag_output_asqa
Model cache directory: c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\models

Loading models...
Loaded from cache: sentence-transformers--all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9787.12it/s]
BertModel LOAD REPORT from: ..\models\sentence-transformers--all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded from cache: google--flan-t5-base


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3510.54it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Models loaded successfully!



## 5. Train Retriever Model

In [7]:
# Create retriever training examples
retriever_examples = loader.create_retriever_examples()
print(f"Created {len(retriever_examples)} retriever training examples")

Creating retriever training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 32409.00it/s]

Created 3450 retriever training examples
Created 3450 retriever training examples


In [8]:
# Train retriever
# Re-running this cell after a crash will auto-resume from the last checkpoint.
print("Training retriever on ASQA dataset...")
rag.train_retriever(
    epochs=config.retriever_epochs,
    batch_size=config.retriever_batch_size,
    lr=config.retriever_lr,
    resume_from_checkpoint=True,
)

Training retriever on ASQA dataset...

Training Retriever
Creating retriever training examples...


Processing: 100%|██████████| 4353/4353 [00:00<00:00, 33680.83it/s]
c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\notebooks\..\src\training\retriever_trainer.py:304: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler(_AMP_DEVICE, enabled=use_fp16)


Created 3450 retriever training examples
Retriever trainer initialized on cpu
Created DataLoader: 3450 examples, batch_size=16
Resuming retriever from checkpoint: rag_output_asqa\checkpoints\retriever_checkpoint.pt


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5363.56it/s]


   Resumed at epoch=6, step=0, best_loss=0.0021

Starting retriever training...
   Epochs: 5
   Learning rate: 2e-05
   Temperature: 20.0
   Accumulation steps: 2
   Mixed precision: True
   Checkpoint every 100 steps → rag_output_asqa\checkpoints

Training complete! Best loss: 0.0021
Model saved to: ..\models\retriever_trained



## 6. Evaluate Retriever

In [9]:
# Evaluate retriever on dev set
print("Evaluating retriever...")
retriever_metrics = rag.evaluate_retriever()

print("\nRetriever Metrics:")
for metric, value in retriever_metrics.items():
    print(f"  {metric}: {value:.4f}")

Evaluating retriever...

Evaluating Retriever

Evaluating retriever on 948 questions...
   Top-K values: [1, 3, 5]
Building corpus from validation set...
   Corpus size: 2010 unique documents
Loading corpus embeddings from cache: rag_output_asqa\corpus_embeds.pt

Evaluating retrieval performance...


Evaluating:   0%|          | 0/948 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 948/948 [00:05<00:00, 162.26it/s]


Retriever Evaluation Results:
   recall@1            : 0.6308
   precision@1         : 0.6308
   recall@3            : 0.7932
   precision@3         : 0.3688
   recall@5            : 0.8249
   precision@5         : 0.2458
   mrr                 : 0.7132


Retriever Metrics:
  recall@1: 0.6308
  precision@1: 0.6308
  recall@3: 0.7932
  precision@3: 0.3688
  recall@5: 0.8249
  precision@5: 0.2458
  mrr: 0.7132


## 7. Build FAISS Index

In [10]:
# Build corpus and index
print("Building corpus and FAISS index...")
rag.build_index()

Building corpus and FAISS index...

Building FAISS Index
Built corpus: 6269 unique passages from 4353 questions
Building FAISS index from 6269 documents...


Encoding: 100%|██████████| 98/98 [00:44<00:00,  2.22it/s]

Built FAISS index: 6269 documents, 384 dimensions
Saved index to rag_output_asqa\index
   - index.faiss: rag_output_asqa\index\index.faiss
   - meta.json: rag_output_asqa\index\meta.json



## 8. Train Generator Model

In [11]:
import gc

# Free the retriever encoder from RAM before loading the generator.
# Both models cannot comfortably coexist on CPU with limited memory.
if hasattr(rag, "retriever_trainer") and rag.retriever_trainer is not None:
    del rag.retriever_trainer.model
    rag.retriever_trainer = None
gc.collect()
print("Freed retriever encoder memory — ready for generator training")

Freed retriever encoder memory — ready for generator training


In [12]:
# Preview available training size (train_generator creates examples internally)
total_available = len(loader.df_train)
print(f"Training set size: {total_available} examples available")
print("Will use up to 2000 examples for generator training")

Training set size: 4353 examples available
Will use up to 2000 examples for generator training


In [13]:
# Train generator
# Re-running this cell after a crash will auto-resume from the last checkpoint.
print("Training generator on ASQA dataset...")
print("Note: This may take several hours for long-form generation")

rag.train_generator(
    epochs=config.generator_epochs,
    batch_size=config.generator_batch_size,
    lr=config.generator_lr,
    max_examples=2000,            # Cap for memory safety
    resume_from_checkpoint=True,  # Auto-resume from last checkpoint on crash
)

Training generator on ASQA dataset...
Note: This may take several hours for long-form generation

Training Generator
Creating generator training examples...


Processing:  46%|████▌     | 2000/4353 [00:00<00:00, 56938.31it/s]

Created 2000 generator training examples
Generator trainer initialized on cpu
Created DataLoader: 2000 examples, batch_size=4
Resuming generator from checkpoint: rag_output_asqa\checkpoints\generator_checkpoint.pt



Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3324.52it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


   Resumed at epoch=4, step=0, global_step=1500

Starting generator training...
   Epochs: 3
   Learning rate: 5e-05
   Warmup ratio: 0.1
   Gradient accumulation: 1
   Max input tokens: 512
   Max target tokens: 128
   Gradient checkpointing: enabled
   Checkpoint every 200 steps → rag_output_asqa\checkpoints


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]

Model and tokenizer saved to ..\models\generator_trained

Training complete!
Model saved to: ..\models\generator_trained



## 9. Baseline 1: Normal RAG Inference

In [14]:
# Test on a sample question
test_question = df_dev.iloc[0]['question']
print(f"Test Question: {test_question}")

answer, contexts = rag.answer(test_question, return_contexts=True)
print(f"\nGenerated Answer:\n{answer}")
print(f"\nRetrieved {len(contexts)} contexts")

Test Question: Who has the highest goals in world football?
QA Pipeline initialized on cpu

Generated Answer:
Josef Bican has scored the most football goals in their career in history. Out of all the active players, Cristiano Ronaldo scored the most goals in his career with over 780 official senior career goals. Ali Daei of Iran has scored the most men's international goals in his career. He is the only player to score over 100 goals in international football with 109 goals. Christine Sinclair is the world's all-time leader for international goals scored for men or women with 187 goals, and is one of the most-capped active international footballers with 300 caps.

Retrieved 5 contexts


In [15]:
# Run inference on full dev set
print("Running Normal RAG inference on dev set...")

normal_rag_results = []

for idx, row in tqdm(df_dev.iterrows(), total=len(df_dev), desc="Normal RAG"):
    question = row['question']
    gold_answer = row['answer']
    
    try:
        # Generate answer
        predicted_answer, contexts = rag.answer(question, return_contexts=True)
        
        normal_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': predicted_answer,
            'contexts': contexts,
            'num_contexts': len(contexts)
        })
    except Exception as e:
        print(f"Error on question {idx}: {e}")
        normal_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': f"ERROR: {str(e)}",
            'contexts': [],
            'num_contexts': 0
        })

# Save results
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

df_normal_rag = pd.DataFrame(normal_rag_results)
df_normal_rag.to_csv(results_dir / "asqa_normal_rag_predictions.csv", index=False)
print(f"\nâœ… Saved Normal RAG results: {len(df_normal_rag)} predictions")

Running Normal RAG inference on dev set...


Normal RAG:   0%|          | 0/948 [00:00<?, ?it/s]

QA Pipeline initialized on cpu


Normal RAG:   0%|          | 1/948 [00:03<1:02:15,  3.94s/it]

QA Pipeline initialized on cpu


...

QA Pipeline initialized on cpu


Normal RAG:   2%|▏         | 15/948 [00:45<46:25,  2.99s/it]

QA Pipeline initialized on cpu


Normal RAG:   2%|▏         | 15/948 [00:47<48:56,  3.15s/it]


KeyboardInterrupt: 

## 10. Baseline 2: RAG + LLM Filter

In [16]:
# Initialize LLM filter pipeline
filter_pipeline = LLMFilterPipeline(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model_name="gemini-2.5-flash",
    context_threshold=6.0,
    answer_threshold=6.0,
    max_concurrent=10
)

print("Initialized LLM filter pipeline")

Initialized LLM filter pipeline


In [17]:
# Test filtering on a sample
test_question = df_dev.iloc[0]['question']
test_answer, test_contexts = rag.answer(test_question, return_contexts=True)

print(f"Question: {test_question}")
print(f"\nOriginal contexts: {len(test_contexts)}")

# Filter contexts
filtered_contexts, context_results = filter_pipeline.filter_contexts(test_question, test_contexts)
print(f"Filtered contexts: {len(filtered_contexts)}")

# Show context filtering results
for i, result in enumerate(context_results):
    print(f"\nContext {i+1}: Score={result.score:.1f}, Relevant={result.is_relevant}")
    print(f"  Reasoning: {result.reasoning}")

# Filter answer
answer_result = filter_pipeline.filter_answer(test_question, test_answer, test_contexts)
print(f"\nAnswer Evaluation:")
print(f"  Faithfulness: {answer_result.faithfulness_score:.1f}")
print(f"  Relevance: {answer_result.relevance_score:.1f}")
print(f"  Completeness: {answer_result.completeness_score:.1f}")
print(f"  Overall: {answer_result.overall_quality}")
print(f"  Reasoning: {answer_result.reasoning}")

QA Pipeline initialized on cpu
Question: Who has the highest goals in world football?

Original contexts: 5
Filtered contexts: 4

Context 1: Score=10.0, Relevant=True
  Reasoning: The context is highly relevant and helpful, providing the top goal scorers across different categories (all-time, active, men's international, overall international) which directly answers the question's most likely interpretation.

Context 2: Score=10.0, Relevant=True
  Reasoning: The context is highly relevant and directly provides specific answers for different interpretations of "highest goals" (all-time, active, men's international, and overall international).

Context 3: Score=10.0, Relevant=True
  Reasoning: The context directly answers who holds the record for the most goals in various categories (all-time career, active career, men's international, and overall international), making it highly relevant and helpful for the ambiguous question.

Context 4: Score=9.0, Relevant=True
  Reasoning: The contex

In [18]:
# Run filtered RAG on dev set
print("Running RAG + LLM Filter on dev set...")
print("Note: This will take longer due to LLM filtering")

filtered_rag_results = []

for idx, row in tqdm(df_dev.iterrows(), total=len(df_dev), desc="Filtered RAG"):
    question = row['question']
    gold_answer = row['answer']
    
    try:
        # Step 1: Retrieve contexts
        contexts, _, _ = rag.indexer.search(question, top_k=config.top_k)
        
        # Step 2: Filter contexts
        filtered_contexts, context_filter_results = filter_pipeline.filter_contexts(
            question, contexts
        )
        
        # Step 3: Generate answer with filtered contexts
        if filtered_contexts:
            qa_pipeline = rag.create_qa_pipeline()
            # Use filtered contexts for generation
            predicted_answer = qa_pipeline._generate_answer(question, filtered_contexts)
        else:
            # Fallback to all contexts if all filtered out
            predicted_answer = rag.answer(question)
            filtered_contexts = contexts
        
        # Step 4: Filter answer
        answer_filter_result = filter_pipeline.filter_answer(
            question, predicted_answer, filtered_contexts
        )
        
        filtered_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': predicted_answer,
            'contexts': contexts,
            'filtered_contexts': filtered_contexts,
            'num_contexts': len(contexts),
            'num_filtered_contexts': len(filtered_contexts),
            'answer_quality': answer_filter_result.overall_quality,
            'faithfulness_score': answer_filter_result.faithfulness_score,
            'relevance_score': answer_filter_result.relevance_score,
            'completeness_score': answer_filter_result.completeness_score,
            'filter_reasoning': answer_filter_result.reasoning
        })
    except Exception as e:
        print(f"Error on question {idx}: {e}")
        filtered_rag_results.append({
            'id': row.get('id', f'asqa_{idx}'),
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': f"ERROR: {str(e)}",
            'contexts': [],
            'filtered_contexts': [],
            'num_contexts': 0,
            'num_filtered_contexts': 0,
            'answer_quality': 'BAD',
            'faithfulness_score': 0.0,
            'relevance_score': 0.0,
            'completeness_score': 0.0,
            'filter_reasoning': f"Error: {str(e)}"
        })

# Save results
df_filtered_rag = pd.DataFrame(filtered_rag_results)
df_filtered_rag.to_csv(results_dir / "asqa_filtered_rag_predictions.csv", index=False)
print(f"\nâœ… Saved Filtered RAG results: {len(df_filtered_rag)} predictions")

Running RAG + LLM Filter on dev set...
Note: This will take longer due to LLM filtering


Filtered RAG:   0%|          | 0/948 [00:00<?, ?it/s]

Error evaluating passage: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 22.783749545s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 22
}
]
Error evaluating passage: 429 You exceeded your current quota, please check your p

Filtered RAG:   0%|          | 1/948 [00:03<49:05,  3.11s/it]

QA Pipeline initialized on cpu
Error on question 0: 'QAPipeline' object has no attribute '_generate_answer'


...

Filtered RAG:   9%|▉         | 88/948 [00:45<06:37,  2.16it/s]

Error evaluating passage: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 37.74020124s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 37
}
]
Error evaluating passage: 429 You exceeded your current quota, please check your pl

Filtered RAG:   9%|▉         | 88/948 [00:45<07:26,  1.92it/s]

Error evaluating passage: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 37.413916977s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 37
}
]
Error evaluating passage: 429 You exceeded your current quota, please check your p

KeyboardInterrupt: 

## 11. Quick Comparison

In [ ]:
# Compare filtering statistics
print("Filtering Statistics:")
print(f"\nContext Filtering:")
total_contexts = df_filtered_rag['num_contexts'].sum()
filtered_contexts = df_filtered_rag['num_filtered_contexts'].sum()
filter_rate = (total_contexts - filtered_contexts) / total_contexts * 100
print(f"  Total contexts retrieved: {total_contexts}")
print(f"  Contexts after filtering: {filtered_contexts}")
print(f"  Filter rate: {filter_rate:.1f}%")

print(f"\nAnswer Quality Distribution:")
quality_counts = df_filtered_rag['answer_quality'].value_counts()
print(quality_counts)
print(f"\nGood answers: {quality_counts.get('GOOD', 0) / len(df_filtered_rag) * 100:.1f}%")
print(f"Bad answers: {quality_counts.get('BAD', 0) / len(df_filtered_rag) * 100:.1f}%")

print(f"\nAverage Scores (Filtered RAG):")
print(f"  Faithfulness: {df_filtered_rag['faithfulness_score'].mean():.2f}")
print(f"  Relevance: {df_filtered_rag['relevance_score'].mean():.2f}")
print(f"  Completeness: {df_filtered_rag['completeness_score'].mean():.2f}")

## 12. Sample Outputs Comparison

In [ ]:
# Compare a few examples side by side
for i in range(min(3, len(df_normal_rag))):
    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    print(f"{'='*80}")
    
    print(f"\nQuestion: {df_normal_rag.iloc[i]['question']}")
    
    print(f"\nGold Answer:\n{df_normal_rag.iloc[i]['gold_answer']}")
    
    print(f"\nNormal RAG Answer:\n{df_normal_rag.iloc[i]['predicted_answer']}")
    
    print(f"\nFiltered RAG Answer:\n{df_filtered_rag.iloc[i]['predicted_answer']}")
    print(f"Quality: {df_filtered_rag.iloc[i]['answer_quality']}")
    print(f"Scores: F={df_filtered_rag.iloc[i]['faithfulness_score']:.1f}, "
          f"R={df_filtered_rag.iloc[i]['relevance_score']:.1f}, "
          f"C={df_filtered_rag.iloc[i]['completeness_score']:.1f}")

## Summary

Successfully implemented and compared two RAG baselines on ASQA:

1. **Normal RAG**: Trained retriever + generator on ASQA long-form QA
2. **RAG + LLM Filter**: Added two-stage filtering (context + answer)

Next steps:
- Generate synthetic labeled data for training
- Comprehensive evaluation with RAGAS metrics
- Detailed analysis of filter effectiveness